# ASSET-12: NAME-REF draw, naming sheets and HT-3
Decisions D205 to D214. Runtime: CPU is enough, no GPU. Run the cells in order.
The person who runs this notebook is never a namer (D213). Never show its output to a namer.

In [ ]:
# Cell 0: setup, then one PASS or FAIL line per input
import os, subprocess, sys
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = '/content/banglaindoorvqa-main'
DRIVE = '/content/drive/MyDrive/BanglaIndoorVQA'
SCRIPT = f'{REPO}/scripts/asset12_nameref.py'
TOKEN = userdata.get('GITHUB_TOKEN')
URL = f'https://{TOKEN}@github.com/blkeyd/banglaindoorvqa-main.git'

def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(TOKEN, '***').strip())
    return r.returncode

if not os.path.isdir(REPO):
    git('clone', URL, REPO)
git('-C', REPO, 'pull')
git('-C', REPO, 'fetch', '--tags', '--force')

def run(*args):
    """Run one command of the ASSET-12 script and stream its printout."""
    cmd = [sys.executable, '-u', SCRIPT, *args, '--drive', DRIVE]
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end='')
    return p.returncode

print('PASS' if os.path.isfile(SCRIPT) else 'FAIL', ' scripts/asset12_nameref.py in the repo')
run('check-inputs')

In [ ]:
# Cell 1: self test on invented data (expect: selftest: 48 of 48 checks OK)
run('selftest')

In [ ]:
# Cell 2: dry run of the whole chain on the real inputs, in a temporary folder (expect: ALL PASS)
run('dryrun')

**Before the draw:** N1 = A3 and N2 = A2 (D214). Their 40 houses are removed from the draw.
N2 helped write the lexicon, so the draw records `--lexicon-writer N2`.

In [ ]:
# Cell 3: the draw (D206, D207, D214)
NAMER1, NAMER2 = 'A3', 'A2'    # N1 = A3 (independent), N2 = A2 (helped write the lexicon)
LEXICON_WRITER = 'N2'          # D214: reported separately, never the main human row
ALSO1, ALSO2 = '', ''          # extra house ids, comma separated (none for this team)
run('draw', '--namer1', NAMER1, '--namer2', NAMER2, '--lexicon-writer', LEXICON_WRITER,
    '--also-exclude1', ALSO1, '--also-exclude2', ALSO2)

In [ ]:
# Cell 4: build the two packages (D209, D210)
# Set True only after both namers confirmed the D205 and D214 rules (guide, section "Before you start").
NAMERS_CLEARED = False
if NAMERS_CLEARED:
    run('build', '--namers-cleared')
else:
    print('Not built. Set NAMERS_CLEARED = True once both namers have confirmed the rules.')

In [ ]:
# Cell 5: check a returned sheet (D211). Upload it first to Drive human/HT-3/returned/
ROLE = 'N1'                    # N1 or N2
FILE = f'{DRIVE}/human/HT-3/returned/HT3_{ROLE}.xlsx'
run('check', '--role', ROLE, '--file', FILE)

In [ ]:
# Cell 6: receive the sheet once its check passes (Gate H evidence; never replaced)
run('receive', '--role', ROLE, '--file', FILE)

In [ ]:
# Cell 7: provisional scores, a pipeline check only; never reported (D208)
run('score', '--provisional')

In [ ]:
# Cell 8: what exists so far
run('status')

In [ ]:
# Cell 9: final scores, only after the tag gate-F exists (D208)
run('score', '--final')